# Neural Network regressor with TensorFlow / Keras

## 1. Imports
!pip install scikeras tensorflow

In [19]:
%pip install tensorflow

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder


import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.models import load_model



Note: you may need to restart the kernel to use updated packages.


## 2. data pre - processing

In [21]:
# Load datasets
train_df = pd.read_csv("data/train.csv")
test_df  = pd.read_csv("data/test.csv")

# Drop columns with no variance (from EDA)
drop_cols = ['X11','X93','X107','X233','X235','X268','X289','X290','X293','X297','X330','X347']
train_df = train_df.drop(columns=drop_cols, errors="ignore")
test_df  = test_df.drop(columns=drop_cols, errors="ignore")

# 3) One-Hot Encode categorical columns
categorical_cols = ['X0','X1','X2','X3','X4','X5','X6','X8']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')

# Ohe on train categoricals
ohe_train = ohe.fit_transform(train_df[categorical_cols])
# Ohe on test categoricals
ohe_test  = ohe.transform(test_df[categorical_cols])

# Transformed dataframes (drop original categoricals, join OHE)
train_transformed = train_df.drop(columns=categorical_cols).join(ohe_train)
test_transformed  = test_df.drop(columns=categorical_cols).join(ohe_test)

# Quick checks
print("Shapes")
print("Original train_df shape:    ", train_df.shape)
print("Transformed train_df shape: ", train_transformed.shape)

print("\n Dropped Categorical Columns")
print(categorical_cols)

print("\n New OHE Columns Names")
print(ohe.get_feature_names_out(categorical_cols)[:20])  # show first 20 OHE columns
print(f"Total new OHE columns: {len(ohe.get_feature_names_out(categorical_cols))}")

Shapes
Original train_df shape:     (4209, 366)
Transformed train_df shape:  (4209, 553)

 Dropped Categorical Columns
['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']

 New OHE Columns Names
['X0_a' 'X0_aa' 'X0_ab' 'X0_ac' 'X0_ad' 'X0_af' 'X0_ai' 'X0_aj' 'X0_ak'
 'X0_al' 'X0_am' 'X0_ao' 'X0_ap' 'X0_aq' 'X0_as' 'X0_at' 'X0_au' 'X0_aw'
 'X0_ax' 'X0_ay']
Total new OHE columns: 195


## 3. Define Train / validation sets

In [22]:
# Define X and y from transformed data
feature_cols = [c for c in train_transformed.columns if c not in ['ID', 'y']]
X = train_transformed[feature_cols]
y = train_transformed['y']

# Train/Valid split (80/20, like before)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Shapes:")
print("X_train:", X_train.shape, "X_valid:", X_valid.shape)

Shapes:
X_train: (3367, 551) X_valid: (842, 551)


## 4. Standartization (scaling)
Imortant: Identify which columns to scale (NNs need normalized inputs) 

**Notes** 
- It’s okay if OHE 0/1 columns get scaled; NNs like normalized inputs
- It's possible scale only continuous columns via ColumnTransformer
- The following Single-scaler approach keeps things simple

In [23]:
# Scale features (fit on TRAIN only to avoid  data leakage into validation set)

scaler = StandardScaler()                         # standardize features to zero mean / unit variance
X_train_scaled = scaler.fit_transform(X_train)    # fit ONLY on training data
X_valid_scaled = scaler.transform(X_valid)        # apply same transform to validation data



## 5. Define Hyperparameters (To change or modify easily later)

In [24]:
UNITS_1     = 128         # neurons in hidden layer 1
UNITS_2     = 64          # neurons in hidden layer 2
DROPOUT     = 0.2         # dropout rate
L2          = 1e-4        # L2 regularization
LR          = 1e-3        # learning rate
BATCH_SIZE  = 256
EPOCHS      = 300
PATIENCE_ES = 20          # patience for EarlyStopping
PATIENCE_LR = 10          # patience for LR reduction

## 6. Build a model (Keras) 
This is a sequential model 

In [25]:
model = Sequential()

# Input + Hidden Layer 1
model.add(Dense(
    units=UNITS_1,
    activation='relu',
    input_dim=X_train_scaled.shape[1],          # number of input features
    kernel_regularizer=regularizers.l2(L2)
))
model.add(Dropout(DROPOUT))

# Hidden Layer 2
model.add(Dense(
    units=UNITS_2,
    activation='relu',
    kernel_regularizer=regularizers.l2(L2)
))
model.add(Dropout(DROPOUT))

# Output Layer (for regression: linear activation)
model.add(Dense(
    units=1,
    activation='linear'
))

/Users/elab2/Desktop/AI_PM/mercedes/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Compile the model** 

In [26]:
# --- 6) Compile the model ---
model.compile(
    loss='mse',                                   # optimize MSE
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    metrics=['mae','mse']                         # track MAE & MSE
)


**Good practices** Callbacks 

In [27]:
es = EarlyStopping(
    monitor='val_loss',
    patience=PATIENCE_ES,
    restore_best_weights=True
)
rlr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=PATIENCE_LR,
    min_lr=1e-6
)
ckpt = ModelCheckpoint(
    filepath='best_keras_regressor.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

**Train the model**

In [28]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_valid_scaled, y_valid),
    callbacks=[es, rlr, ckpt],
    verbose=1
)

Epoch 1/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - loss: 9487.7422 - mae: 96.5425 - mse: 9487.7129 - val_loss: 8559.3174 - val_mae: 91.6451 - val_mse: 8559.2861 - learning_rate: 0.0010
Epoch 2/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7674.1792 - mae: 86.5604 - mse: 7674.1475 - val_loss: 6238.9844 - val_mae: 77.8101 - val_mse: 6238.9502 - learning_rate: 0.0010
Epoch 3/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 4884.7192 - mae: 67.9840 - mse: 4884.6831 - val_loss: 2975.1304 - val_mae: 52.3573 - val_mse: 2975.0908 - learning_rate: 0.0010
Epoch 4/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1769.6432 - mae: 37.4936 - mse: 1769.6008 - val_loss: 528.2809 - val_mae: 18.3997 - val_mse: 528.2349 - learning_rate: 0.0010
Epoch 5/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 453.6931 - mae: 16.6710 - mse: 453.6451 - val_loss: 261.5997 - val_mae: 13.0335 - val_mse: 261.5500 - learning_rate: 0.0010
Epoch 6/300
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 358.37

## 7. Evaluate (validation/test split)

In [30]:
# 9) Evaluate on validation split
y_pred = model.predict(X_valid_scaled).ravel()

rmse = float(np.sqrt(mean_squared_error(y_valid, y_pred)))
mae  = float(mean_absolute_error(y_valid, y_pred))
r2   = float(r2_score(y_valid, y_pred))

print("\nKeras DNN Regressor — Tuned (EarlyStopping) on Validation")
print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R²: {r2:.3f}")


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

Keras DNN Regressor — Tuned (EarlyStopping) on Validation
RMSE: 9.848 | MAE: 6.675 | R²: 0.377


## 8. Save Model 
For later load without training again

In [31]:
#Save & Reload (optional)
best_model = load_model('best_keras_regressor.keras')
model.save('keras_regressor_final.keras')